In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, glob, subprocess, hashlib
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
IOT_RAW = config.DATASETS_DIR/'ciciot2023'
IOT_RAW.mkdir(parents=True, exist_ok=True)
print('target:', IOT_RAW)

# --- space check BEFORE downloading 13 GB into Drive ---
st = os.statvfs('/content/drive/MyDrive')
free_gb = st.f_bavail * st.f_frsize / 1e9
print(f'Drive free space: {free_gb:.1f} GB')
print('CIC-IoT-2023 CSV release is roughly 13 GB unpacked.')
if free_gb < 20:
    print('\nWARNING: less than 20 GB free. Options:')
    print('  A. free space / upgrade storage, then download the full release (cleanest provenance)')
    print('  B. download to local Colab disk (/content), subsample, keep only the working frame')
    print('     on Drive. Faster and space-safe, but the raw release is not retained, so the')
    print('     provenance record must state that the working frame was built from a session-local')
    print('     copy pinned by SHA-256 rather than from a retained archive.')


Mounted at /content/drive
target: /content/drive/MyDrive/NIDS_Datasets/ciciot2023
Drive free space: 210.0 GB
CIC-IoT-2023 CSV release is roughly 13 GB unpacked.


In [3]:
# =============================================================================
# Cell 2 - ACQUIRE. Kaggle mirror, matching the acquisition pattern already used
# for CIC-IDS2017 (dhoogla) and the IoMT work. Set STAGE to choose where the raw
# release lands. 'drive' retains the archive (best provenance); 'local' keeps it
# on session disk only (space-safe) and is recorded as such.
# =============================================================================
STAGE = 'local'          # 'drive' or 'local'
KAGGLE_DATASET = 'madhavmalhotra/unb-cic-iot-dataset'   # CSV mirror of the CIC release

STAGE_DIR = (IOT_RAW if STAGE=='drive' else Path('/content/ciciot2023_raw'))
STAGE_DIR.mkdir(parents=True, exist_ok=True)
print('staging to:', STAGE_DIR)

# Kaggle credentials: upload kaggle.json once, or keep it at Drive root.
KJ = DRIVE_ROOT/'kaggle.json'
if KJ.exists():
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.copy(KJ, '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print('kaggle credentials restored from Drive')
else:
    print('kaggle.json not found at', KJ)
    print('Create an API token at kaggle.com/settings and place kaggle.json at your Drive root,')
    print('or upload it now with: from google.colab import files; files.upload()')

r = subprocess.run(['pip','install','-q','kaggle'], capture_output=True, text=True)
r = subprocess.run(['kaggle','datasets','download','-d',KAGGLE_DATASET,'-p',str(STAGE_DIR),'--unzip'],
                   capture_output=True, text=True)
print((r.stdout + r.stderr)[-2000:])

csvs = sorted(glob.glob(str(STAGE_DIR/'**'/'*.csv'), recursive=True))
print(f'\nCSV files present: {len(csvs)}')
if csvs:
    tot = sum(Path(f).stat().st_size for f in csvs)/1e9
    print(f'total CSV size: {tot:.2f} GB')
    print('first 3:', [Path(f).name for f in csvs[:3]])
else:
    print('No CSVs. If the Kaggle route failed, download the CSV directory directly from')
    print('https://www.unb.ca/cic/datasets/iotdataset-2023.html and place the .csv files under')
    print(STAGE_DIR)


staging to: /content/ciciot2023_raw
kaggle.json not found at /content/drive/MyDrive/kaggle.json
Create an API token at kaggle.com/settings and place kaggle.json at your Drive root,
or upload it now with: from google.colab import files; files.upload()
Dataset URL: https://www.kaggle.com/datasets/madhavmalhotra/unb-cic-iot-dataset
License(s): other
unb-cic-iot-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


CSV files present: 162
total CSV size: 12.97 GB
first 3: ['part-00000-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv', 'part-00001-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv', 'part-00002-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv']


In [ ]:
# =============================================================================
# Cell 3 - VERIFY and PIN. Confirms the release is the expected one before any
# analysis depends on it, and records a SHA-256 manifest so the exact bytes used
# are citable. Same two-layer provenance as the CIC-IDS2017 acquisition.
# =============================================================================
assert csvs, 'no CSVs staged; complete cell 2 first'
head = pd.read_csv(csvs[0], nrows=5)
LABEL_COL = next((c for c in head.columns if c.strip().lower() in ('label','labels','attack','class')), None)
print(f'columns: {len(head.columns)} | label column: {LABEL_COL}')
assert LABEL_COL, 'no label column; inspect head.columns'

from collections import Counter
cnt = Counter()
for i,f in enumerate(csvs):
    cnt.update(pd.read_csv(f, usecols=[LABEL_COL])[LABEL_COL].astype(str).str.strip().value_counts().to_dict())
    if (i+1) % 25 == 0: print(f'  scanned {i+1}/{len(csvs)}')
inv = pd.Series(cnt).sort_values(ascending=False)
print(f'\ntotal rows: {inv.sum():,} | distinct labels: {len(inv)}')
print(inv.to_string())

EXPECTED_LABELS = 34
EXPECTED_ROWS_MIN = 40_000_000
print(f'\nlabel count {len(inv)} (published release has {EXPECTED_LABELS})')
print(f'row count {inv.sum():,} (published release has 46M+)')
if len(inv) != EXPECTED_LABELS:
    print('NOTE: label count differs from the published release. Record which mirror and')
    print('      version this is; the paper must cite the exact source used.')
if inv.sum() < EXPECTED_ROWS_MIN:
    print('NOTE: row count is below the full release, so this mirror may be a subset.')
    print('      That is usable but must be disclosed as the version of record.')

man = {'dataset':'ciciot2023','staged_at':str(STAGE_DIR),'retained_on_drive':(STAGE=='drive'),
       'kaggle_dataset':KAGGLE_DATASET,'n_files':len(csvs),
       'total_bytes':int(sum(Path(f).stat().st_size for f in csvs)),
       'label_column':LABEL_COL,'n_labels':int(len(inv)),'n_rows':int(inv.sum()),
       'label_counts':{k:int(v) for k,v in inv.items()},
       'files':[{'name':Path(f).name,'bytes':Path(f).stat().st_size,
                 'sha256':hashlib.sha256(Path(f).read_bytes()).hexdigest()} for f in csvs[:5]],
       'note':'sha256 recorded for the first five files as a spot pin; full manifest is the '
              'file list plus byte counts. If retained_on_drive is false the raw release was '
              'session-local and the working frame fingerprint in nb31 is the reproducible artefact.'}
(config.REPORTS_DIR/'ciciot2023_acquisition_manifest.json').write_text(json.dumps(man,indent=2))
print('\nmanifest written to reports/ciciot2023_acquisition_manifest.json')

if STAGE != 'drive':
    print('\nIMPORTANT: raw release is on session disk and will be lost when the runtime ends.')
    print('Run notebook 31 in THIS session so the working frame is built and persisted to Drive.')
    print(f'Point nb31 at: {STAGE_DIR}')


columns: 47 | label column: label
  scanned 25/162
  scanned 50/162
  scanned 75/162
  scanned 100/162
  scanned 125/162


In [ ]:
# =============================================================================
# Cell 4 - commit the manifest (raw data is never committed; it is gitignored)
# =============================================================================
def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb30a: CIC-IoT-2023 acquisition manifest and label inventory (no coverage computed)')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)
